# SRD Pose Emergency Core 코드리뷰 정리

대상 파일: `srd_pose_emergency_core.py`

이번 노트북은 Pose Core 한 파일만 기준으로 정리했다. nav, stt, 관제노드 설명은 제외하고 **코어가 프레임 1장을 받아 어떤 순서로 판단을 내리는지**에 집중한다.

## 1. 한 줄 요약

이 코어는 **OpenCV BGR frame -> pose 추론 -> observation -> posture -> motion -> trapped -> emergency_level** 순서로 처리한 뒤, `annotated frame`과 `structured result list`를 반환하는 분석 엔진이다.

## 2. 전체 흐름

```text
OpenCV BGR frame
  -> YOLO Pose 추론
  -> 유효 keypoint 필터링
  -> observation 판정
  -> posture 판정
  -> motion 계산 / 분류
  -> trapped 보조 판정
  -> state duration 계산
  -> emergency_level 결정
  -> annotated frame + result list 반환
```

## 3. 코드 동작 설명

### 입력
- `frame`: `np.ndarray`
- 형식: OpenCV 기준 BGR image

### 출력
- `annotated`: `np.ndarray`
- `results`: `List[dict]`
- `results_to_json(results)`: `str`
- `extract_frame_emergency_level(results)`: `str | None`

### 중요
Pose Core 내부에는 topic, service, action이 없다. 외부 노드가 Python API를 직접 호출하는 구조다.

## 4. 핵심 함수

| 함수 | 역할 |
|---|---|
| `AnalyzerConfig` | 임계값, 모델 경로, 시각화 옵션을 모아둔 설정 클래스 |
| `_classify_visibility` | FULL_BODY / UPPER_BODY / PARTIAL / LOW_CONF 판정 |
| `_classify_posture` | shoulder_tilt, head_drop_ratio, torso_angle 기반 자세 판정 |
| `_motion_value`, `_classify_motion` | 프레임 간 keypoint 이동량으로 움직임 계산 |
| `_possible_trapped` | 매몰 의심 여부 계산 |
| `_state_duration` | track_id별 관측 시간, 상태 지속 시간 계산 |
| `_decide` | 최종 emergency level 결정 |
| `_pack_result` | structured result dict 생성 |
| `analyze_frame_with_results` | 메인 진입점 |

## 5. 판정 로직 요약

### Observation
유효 keypoint 수와 상/하체 존재 여부로 `LOW_CONF`, `FULL_BODY`, `UPPER_BODY`, `PARTIAL`을 나눈다.

### Posture
- 전신: `shoulder_tilt`, `head_drop_ratio`, `torso_angle`, `aspect`를 함께 사용
- 상반신: `shoulder_tilt`, `head_drop_ratio` 중심으로 사용

### Motion
이전 프레임과 현재 프레임의 keypoint 차이로 움직임을 정량화하고, `motion_window`로 smoothing 한다.

### Emergency Level
`observation + posture + motion + trapped + state_sec` 조합으로 최종 등급을 만든다.

## 6. 결과 dict 스키마

- `track_id`
- `bbox`
- `observation`
- `posture`
- `motion`
- `emergency_level`
- `trapped`
- `seen_sec`, `state_sec`
- `shoulder_tilt`, `head_drop_ratio`, `torso_angle`
- `motion_smooth`, `motion_upper`, `motion_core`
- `rep_point_px`, `rep_point_method` 등 대표점 필드

## 7. 코드 주석 기준

Python에서는 **싱글쿼트(`'`)로 주석을 쓰지 않는다.**

- 설명 주석: `#`
- 함수/클래스 설명: `"""docstring"""`

코드리뷰에서는 이 기준으로 정리하는 것이 맞다.

In [ ]:
# v0.000
# file: comment_guideline_example.py
# date: 2026-03-11
# changes:
# - 코드리뷰용 주석 예시

# 1) observation 판정
# 2) posture 판정
# 3) motion 계산

def analyze_frame_with_results(frame):
    """프레임 1장을 분석해서 annotated와 result list를 반환한다."""
    pass


## 8. 코드리뷰에서 꼭 말할 것

1. Pose Core는 ROS 노드가 아니라 라이브러리다.
2. 입력은 BGR frame 1장, 출력은 annotated frame과 structured result다.
3. observation이 뒤 로직의 큰 분기 기준이다.
4. posture와 motion은 단발 수치가 아니라 임계값 + smoothing + duration을 함께 본다.
5. 최종 emergency level은 `state_sec`까지 포함한 시간 누적형 판정이다.